# Format dataset builder: one Parquet table, four target formats

Takes the existing dataset and writes **one Parquet file** where the same code unit appears in four
target formats side by side, so each format can be trained and compared on identical rows.
CPU only, no GPU. Upload `dataset_5000_phase2.jsonl` to `/content`, run all cells, download the Parquet.

| Column | Format |
|---|---|
| `tgt_a_code` | **A**: the code exactly as it is in the dataset (control) |
| `tgt_b_canonical` | **B**: docstrings and comments dropped, layout by `ast.unparse` |
| `tgt_c_tree` | **C**: the full AST as a compact S-expression (expressions are nodes too) |
| `tgt_d_stmt_tree` | **D**: statement-level tree; each expression stays one text leaf (shorter than C) |

D is an extra beyond the three discussed: C is likely to be long against a 512-token output, and D
is the same idea at statement level. Ignore the column if C fits.

## What is in the table
- **Rows:** every generate, repair and refactor row, covering functions, classes, async code, and all domains
  (SOLID/design patterns, API, database, DSA, ML, security, ...). One row per (code unit, task).
- **Inputs are identical across formats** (`input_text`, raw code). Only the target column changes.
- **Token lengths** of every target and input with the CodeT5 tokenizer, plus `fits512_*` flags against the 512 limit.
  Nothing is dropped; filter on the flags.
- **`split`** (train/val/test) is assigned per code unit, so the generate, repair and refactor rows made from the same
  code never straddle splits (the earlier per-task random splits could leak targets into training).
- **Evaluation support:** `golden_io_json` and `golden_func` (the function the golden I/O was captured for).
  Only fully-typed top-level plain functions have golden I/O (about 7% of rows).
- **Import statistics:** `n_imports`, `n_unused_imports`, `import_char_share`.

## Rules
- **Docstrings:** dropped in B/C/D, except refactor rows whose goal is "add docstrings".
- **Imports:** unchanged by default. `PRUNE_UNUSED_IMPORTS` (Cell 1) removes unused imports from B/C/D; A stays raw.
  Each unit carries its source file's whole import block, and most of those imports are unused (see audit section 6).

## Checks built in
- Cell 3b self-tests the converters on 16 hand-written snippets (classes, decorators, async, f-strings, generics,
  `match`, bytes/complex literals, ...) and **stops the notebook if any fail**.
- Every row is verified individually: format text -> Python source -> same AST as the original. A failed row gets its
  format column set to null and `ok_*` = False with the reason in `format_errors`; it is never silently wrong.
- Cell 7 writes the Parquet, reads it back, and re-decodes 400 random rows from the saved file.
- Cell 8 is the audit. It reports coverage, round-trip results, token fit by unit type / task / domain, and one
  sample per kind in all four formats. It does not pick a winner.

## Limits
- Units are the generator's 5-60 line snippets, so whole modules and large classes are not in this table.
- Concept tags (`SOLID`, `repository_pattern`, ...) come from name-matching heuristics and are approximate.
- Checked on Python 3.12 and 3.13 (Colab's version); the self-test would flag any version difference.


In [ ]:
# CELL 0 — INSTALL
!pip install -q transformers==4.46.3

In [ ]:
# CELL 1 — IMPORTS + CONFIG
# CPU only. No model is loaded or trained in this notebook.
import ast, copy, hashlib, json, math, os, random, re, sys, time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | pyarrow {pa.__version__}")
assert sys.version_info >= (3, 9), "ast.unparse needs Python 3.9+"

# Input: the Phase 2 file (has golden_io). The earlier dataset_5000_fixed.jsonl also works
# but has no golden_io, so has_golden_io would be False everywhere.
INPUT_CANDIDATES = [
    "/content/dataset_5000_phase2.jsonl",
    "/content/dataset_5000_fixed.jsonl",
]
OUT_DIR = Path("/content/phase3_formats")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / "dataset_formats.parquet"

TOKENIZER_NAME = "Salesforce/codet5-small"   # the Phase 2 target model's tokenizer
MAX_TARGET_TOKENS = 512                      # output budget
MAX_INPUT_TOKENS = 512                       # encoder window
BASE_TASKS = ("generate", "repair", "refactor")
SPLIT_SEED_SALT = "formats-v1"               # changes the split only if you change this

# Imports: each unit in the dataset carries its whole source file's import block, and most of
# those imports are unused by the unit (see audit section 6). False = keep targets exactly as
# they are (what you asked for). True = also drop unused imports from B, C and D (A stays raw).
PRUNE_UNUSED_IMPORTS = False
print("Config ready.")

In [ ]:
# CELL 2 — LOAD THE DATA (robust to a file whose newlines were stripped in transit)
src_path = next((Path(p) for p in INPUT_CANDIDATES if Path(p).exists()), None)
if src_path is None:
    raise FileNotFoundError("Upload one of these into /content first: " + ", ".join(INPUT_CANDIDATES))
print("Reading:", src_path, f"({src_path.stat().st_size/1e6:.1f} MB)")

raw = src_path.read_text()
all_rows = []
for line in raw.split("\n"):
    line = line.strip()
    if line:
        try:
            all_rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass
if len(all_rows) < 10:
    print("Line-by-line parse found almost nothing; using a newline-independent parser.")
    dec, idx, text, all_rows = json.JSONDecoder(), 0, raw.strip(), []
    while idx < len(text):
        while idx < len(text) and text[idx] in " \t\r\n":
            idx += 1
        if idx >= len(text):
            break
        obj, idx = dec.raw_decode(text, idx)
        all_rows.append(obj)
if not all_rows:
    raise ValueError("Could not parse any rows; the file is probably truncated. Re-download and re-upload it.")

rows = [r for r in all_rows if r.get("task") in BASE_TASKS]
print(f"Loaded {len(all_rows)} rows; keeping {len(rows)} generate/repair/refactor rows "
      f"(skeleton/fill rows from the Phase 2 file are not used here).")
print("Tasks:", dict(Counter(r["task"] for r in rows)))
n_gold = sum(1 for r in rows if r.get("golden_io"))
print(f"Rows with golden_io in this file: {n_gold}"
      + ("" if n_gold else "   <-- none; this looks like the older file without golden_io"))

need = ["id", "task", "instruction", "target_code", "domain"]
missing = [(r.get("id"), k) for r in rows for k in need if k not in r]
assert not missing, f"rows missing required fields, e.g. {missing[:3]}"

## Format converters (deterministic, no model) and their self-test


In [ ]:
# CELL 3 — FORMAT CONVERTERS: canonical code (B), compact tree (C), statement tree (D)
#
# All three are DETERMINISTIC and checked per row by a round-trip test:
#   format text -> rebuilt Python source -> same AST as the original (docstrings aside).
# No model is involved anywhere in this cell.
#
#   B  canonical code : docstrings/comments dropped, layout by ast.unparse
#   C  compact tree   : the FULL AST (every node, expressions included) as an S-expression
#   D  statement tree : statements as tree nodes, each expression kept as one text leaf
#
# C/D notation (single line, one space between items):
#   (TAG item ...)  node        [item ...]  list        None / True / False / ...   literals
#   10  1.5  -1     numbers     name  os.path  *   bare identifier-like strings
#   "text"  JSON string         x"ff00" bytes           j"2j" complex
#   `n + 1`         expression text leaf (D only; a literal backtick is written as two)
# Dropped from the text and restored on rebuild: ctx, type_comment, kind, type_ignores.
# type_params (generics) is written only when non-empty, as an optional trailing item.

import ast, copy, json, math, re, sys

sys.setrecursionlimit(20000)

TAGS = {
    "Module": "mod", "FunctionDef": "def", "AsyncFunctionDef": "adef", "ClassDef": "class",
    "Return": "ret", "Delete": "del", "Assign": "=", "AugAssign": "aug", "AnnAssign": "ann",
    "For": "for", "AsyncFor": "afor", "While": "while", "If": "if", "With": "with",
    "AsyncWith": "awith", "Raise": "raise", "Try": "try", "TryStar": "trystar",
    "Assert": "assert", "Import": "import", "ImportFrom": "from", "Global": "global",
    "Nonlocal": "nonlocal", "Expr": "expr", "Pass": "pass", "Break": "break",
    "Continue": "continue", "BoolOp": "bool", "NamedExpr": "walrus", "BinOp": "bin",
    "UnaryOp": "un", "Lambda": "lam", "IfExp": "ife", "Dict": "dict", "Set": "set",
    "ListComp": "lc", "SetComp": "sc", "DictComp": "dc", "GeneratorExp": "ge",
    "Await": "await", "Yield": "yield", "YieldFrom": "yieldfrom", "Compare": "cmp",
    "Call": "call", "FormattedValue": "fv", "JoinedStr": "fstr", "Constant": "c",
    "Attribute": "attr", "Subscript": "sub", "Starred": "star", "Name": "n", "List": "list",
    "Tuple": "tup", "Slice": "slice", "comprehension": "comp", "ExceptHandler": "exc",
    "arguments": "args", "arg": "arg", "keyword": "kw", "alias": "as", "withitem": "wi",
    "Add": "+", "Sub": "-", "Mult": "*", "MatMult": "@", "Div": "/", "Mod": "%", "Pow": "**",
    "LShift": "<<", "RShift": ">>", "BitOr": "|", "BitXor": "^", "BitAnd": "&",
    "FloorDiv": "//", "And": "and", "Or": "or", "Invert": "~", "Not": "not", "UAdd": "u+",
    "USub": "u-", "Eq": "==", "NotEq": "!=", "Lt": "<", "LtE": "<=", "Gt": ">", "GtE": ">=",
    "Is": "is", "IsNot": "isnot", "In": "in", "NotIn": "notin",
}
assert len(set(TAGS.values())) == len(TAGS), "duplicate short tag"
TAG_TO_CLS = {tag: getattr(ast, name) for name, tag in TAGS.items() if hasattr(ast, name)}

DROP_FIELDS = {"ctx", "type_comment", "kind", "type_ignores", "type_params"}

def kept_fields(cls):
    return [f for f in cls._fields if f not in DROP_FIELDS]

def _default_for(field):
    if field == "ctx":
        return ast.Load()
    if field in ("type_ignores", "type_params"):
        return []
    return None

def _jstr(s):
    out = json.dumps(s, ensure_ascii=False)
    try:
        out.encode("utf-8")
    except UnicodeEncodeError:
        out = json.dumps(s, ensure_ascii=True)
    return out

_IDENT = re.compile(r"[^\W\d]\w*(\.[^\W\d]\w*)*\Z")

def _is_bare(s):
    return s == "*" or (bool(_IDENT.match(s)) and s not in ("None", "True", "False"))

def _prim(v, quote_str):
    if v is None:
        return "None"
    if v is True:
        return "True"
    if v is False:
        return "False"
    if v is Ellipsis:
        return "..."
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        if math.isnan(v):
            raise ValueError("nan constant")
        if math.isinf(v):
            return "1e999" if v > 0 else "-1e999"
        return repr(v)
    if isinstance(v, complex):
        return "j" + _jstr(repr(v))
    if isinstance(v, bytes):
        return "x" + _jstr(v.hex())
    if isinstance(v, str):
        return _jstr(v) if (quote_str or not _is_bare(v)) else v
    raise TypeError(f"unsupported primitive {type(v)}")

def _ser(v, mode, in_const=False):
    if isinstance(v, ast.AST):
        if mode == "stmt" and isinstance(v, ast.expr):
            return "`" + ast.unparse(v).replace("`", "``") + "`"
        cls = type(v)
        parts = [TAGS.get(cls.__name__, cls.__name__)]
        is_const = isinstance(v, ast.Constant)
        for f in kept_fields(cls):
            parts.append(_ser(getattr(v, f, None), mode, is_const))
        if "type_params" in cls._fields and getattr(v, "type_params", None):
            parts.append(_ser(v.type_params, mode, False))   # generics: written only when present
        return "(" + " ".join(parts) + ")"
    if isinstance(v, list):
        return "[" + " ".join(_ser(x, mode, in_const) for x in v) + "]"
    return _prim(v, in_const)

def tree_to_text(tree, mode):
    assert mode in ("full", "stmt")
    return _ser(tree, mode)

# ---------- text -> tokens -> nested values -> AST -> source ----------
_STOP = set(' \t\r\n()[]"`')

def _read_jstr(s, i):
    j = i + 1
    while j < len(s):
        if s[j] == "\\":
            j += 2
            continue
        if s[j] == '"':
            return json.loads(s[i:j + 1]), j + 1
        j += 1
    raise ValueError("unterminated string")

def _lex(s):
    i, n, toks = 0, len(s), []
    while i < n:
        ch = s[i]
        if ch in " \t\r\n":
            i += 1
        elif ch in "()[]":
            toks.append(ch)
            i += 1
        elif ch == "`":
            i += 1
            buf = []
            while True:
                if i >= n:
                    raise ValueError("unterminated expression text")
                if s[i] == "`":
                    if i + 1 < n and s[i + 1] == "`":
                        buf.append("`")
                        i += 2
                        continue
                    i += 1
                    break
                buf.append(s[i])
                i += 1
            toks.append(("expr", "".join(buf)))
        elif ch == '"':
            val, i = _read_jstr(s, i)
            toks.append(("str", val))
        else:
            j = i
            while j < n and s[j] not in _STOP:
                j += 1
            word = s[i:j]
            if j < n and s[j] == '"' and word in ("x", "j"):
                val, j2 = _read_jstr(s, j)
                toks.append(("bytes", bytes.fromhex(val)) if word == "x" else ("complex", complex(val)))
                i = j2
            else:
                toks.append(("bare", word))
                i = j
    return toks

_INT = re.compile(r"-?\d+\Z")
_FLOAT = re.compile(r"-?(\d+\.\d*|\.\d+|\d+)([eE][+-]?\d+)?\Z")

def _bare_to_prim(w):
    if w == "None":
        return None
    if w == "True":
        return True
    if w == "False":
        return False
    if w == "...":
        return Ellipsis
    if _INT.match(w):
        return int(w)
    if _FLOAT.match(w):
        return float(w)
    return w

def _parse(toks, pos):
    if pos >= len(toks):
        raise ValueError("unexpected end of text")
    t = toks[pos]
    if t == "(":
        pos += 1
        if pos >= len(toks) or not (isinstance(toks[pos], tuple) and toks[pos][0] == "bare"):
            raise ValueError("node without a tag")
        tag = toks[pos][1]
        pos += 1
        args = []
        while pos < len(toks) and toks[pos] != ")":
            v, pos = _parse(toks, pos)
            args.append(v)
        if pos >= len(toks):
            raise ValueError("unclosed node")
        return ("node", tag, args), pos + 1
    if t == "[":
        pos += 1
        items = []
        while pos < len(toks) and toks[pos] != "]":
            v, pos = _parse(toks, pos)
            items.append(v)
        if pos >= len(toks):
            raise ValueError("unclosed list")
        return ("list", items), pos + 1
    if isinstance(t, tuple):
        kind, val = t
        if kind == "expr":
            return ("expr", val), pos + 1
        if kind == "bare":
            return ("prim", _bare_to_prim(val)), pos + 1
        return ("prim", val), pos + 1
    raise ValueError(f"unexpected token {t!r}")

def _parse_expr(text):
    try:
        return ast.parse("(" + text + "\n)", mode="eval").body
    except SyntaxError:
        # a lone starred item such as  *bases  is only legal inside a call
        call = ast.parse("_f(" + text + "\n)", mode="eval").body
        if len(call.args) == 1 and not call.keywords:
            return call.args[0]
        raise

def _cls_for(tag):
    cls = TAG_TO_CLS.get(tag)
    if cls is None:
        cls = getattr(ast, tag, None)
        if not (isinstance(cls, type) and issubclass(cls, ast.AST)):
            raise ValueError(f"unknown tag {tag!r}")
    return cls

def _build(v):
    kind = v[0]
    if kind == "list":
        return [_build(x) for x in v[1]]
    if kind == "prim":
        return v[1]
    if kind == "expr":
        return _parse_expr(v[1])
    _, tag, args = v
    cls = _cls_for(tag)
    fields = kept_fields(cls)
    has_tp = "type_params" in cls._fields and len(args) == len(fields) + 1
    if len(args) != len(fields) and not has_tp:
        raise ValueError(f"{tag}: expected {len(fields)} items, got {len(args)}")
    kwargs = {f: _build(a) for f, a in zip(fields, args)}
    if has_tp:
        kwargs["type_params"] = _build(args[-1])
    for f in cls._fields:
        if f not in kwargs:
            kwargs[f] = _default_for(f)
    return cls(**kwargs)

def text_to_source(text):
    """Deterministic render: tree text -> Python source (no model involved)."""
    toks = _lex(text)
    v, pos = _parse(toks, 0)
    if pos != len(toks):
        raise ValueError("trailing text after the tree")
    tree = ast.fix_missing_locations(_build(v))
    return ast.unparse(tree)

# ---------- docstring policy + canonical code + per-row checks ----------
def strip_docstrings(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.Module, ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)):
            b = node.body
            if b and isinstance(b[0], ast.Expr) and isinstance(b[0].value, ast.Constant) \
               and isinstance(b[0].value.value, str):
                node.body = b[1:] or [ast.Pass()]
    return tree

def norm_dump(x):
    tree = ast.parse(x) if isinstance(x, str) else copy.deepcopy(x)
    for n in ast.walk(tree):
        if isinstance(n, ast.Constant):
            n.kind = None
        if hasattr(n, "type_comment"):
            n.type_comment = None
    return ast.dump(tree)

def _check(name, fn):
    try:
        return fn(), None
    except Exception as e:
        return None, f"{name}: {type(e).__name__}: {str(e)[:120]}"

def used_names(tree):
    """Names the unit's own code refers to (excluding its import lines). Conservative: any word
    inside a string literal also counts, so imports used only in string annotations are kept."""
    used = set()
    for n in tree.body:
        if isinstance(n, (ast.Import, ast.ImportFrom)):
            continue
        for x in ast.walk(n):
            if isinstance(x, ast.Name):
                used.add(x.id)
            elif isinstance(x, ast.Constant) and isinstance(x.value, str):
                used.update(re.findall(r"[A-Za-z_]\w*", x.value))
    return used

def _bound_names(node):
    if isinstance(node, ast.Import):
        return [a.asname or a.name.split(".")[0] for a in node.names]
    return [a.asname or a.name for a in node.names if a.name != "*"]

def import_stats(tree, code):
    """(imports, unused imports, share of characters that are import lines) for a parsed unit."""
    used = used_names(tree)
    lines = code.split("\n")
    total = unused = chars = 0
    for n in tree.body:
        if isinstance(n, (ast.Import, ast.ImportFrom)):
            chars += sum(len(lines[i]) + 1 for i in range(n.lineno - 1, n.end_lineno))
            if isinstance(n, ast.ImportFrom) and n.module == "__future__":
                continue
            for b in _bound_names(n):
                total += 1
                unused += b not in used
    return total, unused, chars / max(1, len(code))

def prune_unused_imports(tree):
    """Drop imported names the rest of the unit never uses. Keeps __future__ and star imports;
    prunes nothing if the code uses globals()/eval()/exec()/vars()/__all__. Returns names removed."""
    used = used_names(tree)
    if used & {"globals", "locals", "eval", "exec", "vars", "__all__"}:
        return 0
    removed, new_body = 0, []
    for n in tree.body:
        if isinstance(n, ast.Import):
            keep = [a for a in n.names if (a.asname or a.name.split(".")[0]) in used]
        elif isinstance(n, ast.ImportFrom) and n.module != "__future__" and not any(a.name == "*" for a in n.names):
            keep = [a for a in n.names if (a.asname or a.name) in used]
        else:
            new_body.append(n)
            continue
        removed += len(n.names) - len(keep)
        if keep:
            n.names = keep
            new_body.append(n)
    tree.body = new_body or [ast.Pass()]
    return removed

def make_formats(code, keep_docstrings=False, prune_imports=False):
    """Build B, C, D for one piece of code and verify each by round-trip.
    prune_imports=True also removes unused imports from B/C/D (A is never changed).
    A bad format is returned as None with ok=False and an error string, never silently wrong."""
    res = {"b": None, "c": None, "d": None, "ok_b": False, "ok_c": False, "ok_d": False,
           "err": [], "n_pruned": 0}
    tree = ast.parse(code)
    base = copy.deepcopy(tree)
    if not keep_docstrings:
        strip_docstrings(base)
    if prune_imports:
        res["n_pruned"] = prune_unused_imports(base)
    ref = norm_dump(base)

    b_text, e = _check("B", lambda: ast.unparse(base))
    if b_text is not None:
        ok, e = _check("B", lambda: (norm_dump(b_text) == ref) and (compile(b_text, "<b>", "exec") is not None))
        res["b"], res["ok_b"] = b_text, bool(ok)
        if not ok:
            res["b"] = None
            e = e or "B: round-trip mismatch"
    if e:
        res["err"].append(e)

    for key, mode in (("c", "full"), ("d", "stmt")):
        text, e = _check(key.upper(), lambda m=mode: tree_to_text(base, m))
        if text is not None:
            rebuilt, e = _check(key.upper(), lambda t=text: text_to_source(t))
            if rebuilt is not None:
                ok, e = _check(key.upper(), lambda r=rebuilt: norm_dump(r) == ref)
                if ok:
                    res[key], res["ok_" + key] = text, True
                else:
                    e = e or f"{key.upper()}: round-trip mismatch"
        if e:
            res["err"].append(e)
    return res

print("Converters ready (B canonical code, C compact tree, D statement tree).")

In [ ]:
# CELL 3b — SELF-TEST of the converters (the notebook stops here if anything fails)
SNIPPETS = {
"function": '''def f(a: int, b: int = 2) -> int:
    """Add."""
    return a + b
''',
"class_abc_property": '''from abc import ABC, abstractmethod
class Repo(ABC):
    """Interface."""
    @abstractmethod
    def get(self, key: str) -> "Item | None": ...
    @property
    def size(self):
        return len(self._items)
    @staticmethod
    def make(*args, **kwargs):
        return Repo(*args, **kwargs)
''',
"decorators_args": '''import functools
def retry(times=3):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*a, **k):
            for i in range(times):
                try:
                    return fn(*a, **k)
                except (ValueError, KeyError) as e:
                    last = e
            raise RuntimeError("failed") from last
        return wrapper
    return deco
''',
"async_code": '''import asyncio
async def fetch(session, urls):
    async with session.get(urls[0]) as r:
        async for chunk in r.content:
            await asyncio.sleep(0)
    return await r.text()
''',
"fstrings": '''def g(x, name):
    return f"{name!r:>10} = {x:.2f} {{literal}} {x + 1}"
''',
"comprehensions_lambda_walrus": '''def h(xs):
    a = [x * 2 for x in xs if x > 0]
    b = {k: v for k, v in zip(xs, xs)}
    c = {x for x in xs}
    d = sum(int(c) for c in "123")
    f = lambda y, z=1: y if y > z else z
    if (n := len(xs)) > 3:
        return n
    return a, b, c, d, f
''',
"try_with_global": '''counter = 0
def k(path):
    global counter
    with open(path) as fh, open(path + ".bak", "w") as out:
        try:
            data = fh.read()
        except OSError:
            data = ""
        else:
            counter += 1
        finally:
            out.close()
    assert data is not None, "no data"
    del data
''',
"literals": '''def lit():
    return [b"\\xff\\xd8", 3j, ..., 10**20, -5, 1.5e-07, 1e999, "tab\\t quote\\" back`tick", None, True, False, 0x1F]
''',
"slices_star_assign": '''def s(xs):
    a, *b = xs
    for *c, d in [xs]:
        pass
    return xs[1:3], xs[::2], xs[0], (*a, 1), {**{}, "k": 1}
''',
"conditions": '''def cond(a, b, c):
    x = a if b else c
    y = a < b <= c != 3
    z = not a and (b or c)
    w = a is not None and b not in [1, 2]
    return -a, +b, ~c, x, y, z, w
''',
"unused_imports": '''import os, sys
from typing import List, Optional
from collections import OrderedDict as OD
def f(x: "Optional[int]") -> int:
    return sys.maxsize
''',
"imports": '''from __future__ import annotations
import os.path as osp, sys
from . import sibling
from ..pkg.mod import name as alias, other
from typing import *
''',
"nested_class_dataclass": '''from dataclasses import dataclass, field
@dataclass(frozen=True)
class Order:
    id: int
    items: list = field(default_factory=list)
    class Meta:
        ordering = ("id",)
    def total(self) -> float:
        return sum(i.price for i in self.items)
''',
"star_bases_kwonly": '''class A(*bases, metaclass=Meta):
    pass
def m(a, /, b, *, c, **kw):
    pass
''',
}
if sys.version_info >= (3, 12):
    SNIPPETS["generics_pep695"] = '''def first[T: Comparable](xs: list[T]) -> T:
    return xs[0]
class Box[T]:
    def get(self) -> T: ...
'''
if sys.version_info >= (3, 10):
    SNIPPETS["match_statement"] = '''def mt(cmd):
    match cmd:
        case ["go", direction]:
            return direction
        case {"k": v} if v > 1:
            return v
        case Point(x=0, y=0) | None:
            return 0
        case _:
            return -1
'''

failures = []
for name, code in SNIPPETS.items():
    for keep, prune in ((False, False), (True, False), (False, True)):
        r = make_formats(code, keep_docstrings=keep, prune_imports=prune)
        status = f"B={r['ok_b']} C={r['ok_c']} D={r['ok_d']}"
        if not (r["ok_b"] and r["ok_c"] and r["ok_d"]):
            failures.append((name, keep, prune, r["err"]))
    print(f"  {name:30s} {status}")

if failures:
    for name, keep, prune, err in failures:
        print(f"FAILED  {name} keep_docstrings={keep} prune={prune}: {err}")
    raise AssertionError(f"{len(failures)} self-test case(s) failed — do not continue until fixed")
print(f"\nSelf-test passed: {len(SNIPPETS)} snippets x 3 settings (docstrings dropped / kept / imports pruned), all three formats round-trip.")

demo = make_formats(SNIPPETS["function"], keep_docstrings=False)
print("\nExample (function, docstring dropped):")
print("B:\n" + demo["b"])
print("C:", demo["c"])
print("D:", demo["d"])

pr = make_formats(SNIPPETS["unused_imports"], prune_imports=True)
assert pr["n_pruned"] == 3 and "os" not in pr["b"] and "OrderedDict" not in pr["b"] and "import sys" in pr["b"], pr
assert "Optional" in pr["b"], "an import used only in a string annotation must be kept"
print("\nImport pruning self-test passed (removed the 3 unused names, kept sys and the string-annotation use of Optional).")

## Build the table, count tokens, split


In [ ]:
# CELL 4 — BUILD THE TABLE ROWS: all four target formats side by side, with checks
#
# One row = one (code unit, task). The generate, repair and refactor rows made from the
# same code share a unit_id, so they can be kept in the same split (no leakage).
# Encoder inputs are IDENTICAL across formats; only the target column changes.
#   A  tgt_a_code        the code exactly as it is in the dataset
#   B  tgt_b_canonical   docstrings/comments dropped, layout by ast.unparse
#   C  tgt_c_tree        full compact AST (expressions are nodes too)
#   D  tgt_d_stmt_tree   statement tree, each expression kept as one text leaf
# Docstring rule for B/C/D: dropped everywhere EXCEPT refactor rows whose goal is
# "add docstrings" (there the docstring is the point of the task).
# Import rule: unchanged unless PRUNE_UNUSED_IMPORTS is True (Cell 1).

def input_text_for(r):
    t = r["task"]
    if t == "generate":
        return f"<GENERATE> {r['instruction']}"
    if t == "repair":
        return (f"<REPAIR> {r['instruction']}\n<CODE>\n{r.get('context_code','')}\n</CODE>"
                f"\n<ERROR>\n{r.get('error','')}\n</ERROR>")
    return f"<REFACTOR> {r['instruction']}\n<CODE>\n{r.get('context_code','')}\n</CODE>"

def unit_info(code):
    tree = ast.parse(code)
    kinds = [type(n).__name__ for n in tree.body]
    if "ClassDef" in kinds:
        utype = "class"
    elif "AsyncFunctionDef" in kinds:
        utype = "async_function"
    elif "FunctionDef" in kinds:
        utype = "function"
    else:
        utype = "other"
    n_top = sum(k in ("ClassDef", "FunctionDef", "AsyncFunctionDef") for k in kinds)
    n_methods = sum(isinstance(m, (ast.FunctionDef, ast.AsyncFunctionDef))
                    for c in tree.body if isinstance(c, ast.ClassDef) for m in c.body)
    return utype, n_top, n_methods

def golden_func_name(code):
    """Same rule the Phase 2 golden_io capture used: first top-level, non-async function whose
    parameters are all annotated. Stored so evaluation calls THIS function (not an imported name)."""
    for node in ast.iter_child_nodes(ast.parse(code)):
        if isinstance(node, ast.FunctionDef):
            args = [a for a in node.args.args if a.arg != "self"]
            if len(args) == len(node.args.args) and args and all(a.annotation is not None for a in args):
                return node.name
    return None

records, err_kinds = [], Counter()
t0 = time.time()
for r in rows:
    code = r["target_code"]
    keep = (r["task"] == "refactor") and ("docstring" in r["instruction"].lower())
    unit_id = hashlib.sha1(code.encode("utf-8")).hexdigest()[:12]
    try:
        utype, n_top, n_methods = unit_info(code)
        f = make_formats(code, keep_docstrings=keep, prune_imports=PRUNE_UNUSED_IMPORTS)
        n_imp, n_unused, imp_share = import_stats(ast.parse(code), code)
    except Exception as e:
        utype, n_top, n_methods = "other", 0, 0
        n_imp, n_unused, imp_share = 0, 0, 0.0
        f = {"b": None, "c": None, "d": None, "ok_b": False, "ok_c": False, "ok_d": False,
             "n_pruned": 0, "err": [f"A: {type(e).__name__}: {str(e)[:100]}"]}
    for e in f["err"]:
        err_kinds[e[:60]] += 1
    gold = r.get("golden_io")
    records.append({
        "id": r["id"], "unit_id": unit_id,
        "task": r["task"], "domain": r.get("domain", ""), "difficulty": r.get("difficulty", ""),
        "source_repo": r.get("source_repo", ""),
        "unit_type": utype, "n_lines": len(code.strip().split("\n")),
        "n_top_level_defs": n_top, "n_methods": n_methods,
        "concepts": list(r.get("concepts", []) or []), "libraries": list(r.get("libraries", []) or []),
        "instruction": r["instruction"], "context_code": r.get("context_code", "") or "",
        "error": r.get("error", "") or "", "input_text": input_text_for(r),
        "tgt_a_code": code, "tgt_b_canonical": f["b"], "tgt_c_tree": f["c"], "tgt_d_stmt_tree": f["d"],
        "ok_b": f["ok_b"], "ok_c": f["ok_c"], "ok_d": f["ok_d"],
        "docstrings_kept": keep, "imports_pruned": bool(PRUNE_UNUSED_IMPORTS), "n_imports_removed": f["n_pruned"],
        "n_imports": n_imp, "n_unused_imports": n_unused, "import_char_share": round(imp_share, 4),
        "format_errors": " | ".join(f["err"]),
        "has_golden_io": bool(gold),
        "golden_func": golden_func_name(code) if gold else None,
        "golden_io_json": json.dumps(gold, ensure_ascii=False) if gold else None,
    })

df = pd.DataFrame(records)
print(f"Built {len(df)} rows in {time.time()-t0:.0f}s")
for k in "bcd":
    n_ok = int(df[f"ok_{k}"].sum())
    print(f"  round-trip OK, format {k.upper()}: {n_ok}/{len(df)} ({n_ok/len(df)*100:.2f}%)")
if err_kinds:
    print("  error kinds:", dict(err_kinds.most_common(5)))
bad_gold = df[df.has_golden_io & df.golden_func.isna()]
print(f"  golden_io rows with no identifiable function: {len(bad_gold)} (should be 0)")

In [ ]:
# CELL 5 — TOKEN LENGTHS with the CodeT5 tokenizer (the real 512-token budget)
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
tok.model_max_length = 10**7          # silence the "longer than 512" warning; we only count
print("Tokenizer:", type(tok).__name__, "| fast:", getattr(tok, "is_fast", None))

def count_tokens(texts, chunk=1000):
    """Length including the special tokens, i.e. exactly what a training label/input would occupy."""
    out = []
    for i in range(0, len(texts), chunk):
        enc = tok(texts[i:i + chunk], add_special_tokens=True, truncation=False)["input_ids"]
        out.extend(len(x) for x in enc)
    return out

def count_col(col):
    vals = df[col].tolist()
    idx = [i for i, v in enumerate(vals) if isinstance(v, str)]
    lens = count_tokens([vals[i] for i in idx])
    full = [None] * len(vals)
    for i, l in zip(idx, lens):
        full[i] = l
    return pd.array(full, dtype="Int64")

t0 = time.time()
df["tok_input"] = count_col("input_text")
df["tok_a"] = count_col("tgt_a_code")
df["tok_b"] = count_col("tgt_b_canonical")
df["tok_c"] = count_col("tgt_c_tree")
df["tok_d"] = count_col("tgt_d_stmt_tree")
for k in "abcd":
    df[f"fits512_{k}"] = (df[f"tok_{k}"].notna() & (df[f"tok_{k}"].fillna(10**9) <= MAX_TARGET_TOKENS)).astype(bool)
df["fits512_input"] = (df["tok_input"].fillna(10**9) <= MAX_INPUT_TOKENS).astype(bool)
print(f"Counted tokens in {time.time()-t0:.0f}s")
print("\nShare of rows whose TARGET fits in 512 tokens:")
for k, name in zip("abcd", ["A original", "B canonical", "C full tree", "D stmt tree"]):
    print(f"  {name:12s} {df[f'fits512_{k}'].mean()*100:5.1f}%   (mean {df[f'tok_{k}'].mean():6.0f} tokens, median {df[f'tok_{k}'].median():6.0f})")
print(f"  encoder input fits 512: {df['fits512_input'].mean()*100:.1f}%")

In [ ]:
# CELL 6 — TRAIN / VAL / TEST SPLIT, assigned per code unit
# A unit's generate, repair and refactor rows all have the same target code. If they landed in
# different splits, test targets would appear in training. Hashing unit_id keeps them together.
def split_of(unit_id):
    h = int(hashlib.sha1((SPLIT_SEED_SALT + unit_id).encode()).hexdigest()[:8], 16) % 100
    return "train" if h < 80 else ("val" if h < 90 else "test")

df["split"] = df["unit_id"].map(split_of)
leaks = df.groupby("unit_id")["split"].nunique().gt(1).sum()
assert leaks == 0, f"{leaks} units span more than one split"
print("Split sizes:", df["split"].value_counts().to_dict())
print(pd.crosstab(df["split"], df["task"]).to_string())
print("Units shared across splits: 0 (checked)")

## Write the Parquet file, verify it, audit it, download it


In [ ]:
# CELL 7 — WRITE THE PARQUET FILE AND VERIFY IT BY READING IT BACK
COLUMNS = [
    "id", "unit_id", "split", "task", "domain", "difficulty", "source_repo",
    "unit_type", "n_lines", "n_top_level_defs", "n_methods", "concepts", "libraries",
    "instruction", "context_code", "error", "input_text",
    "tgt_a_code", "tgt_b_canonical", "tgt_c_tree", "tgt_d_stmt_tree",
    "ok_b", "ok_c", "ok_d", "docstrings_kept", "imports_pruned", "n_imports_removed",
    "n_imports", "n_unused_imports", "import_char_share", "format_errors",
    "tok_input", "tok_a", "tok_b", "tok_c", "tok_d",
    "fits512_input", "fits512_a", "fits512_b", "fits512_c", "fits512_d",
    "has_golden_io", "golden_func", "golden_io_json",
]
assert set(COLUMNS) == set(df.columns), f"column mismatch: {set(COLUMNS) ^ set(df.columns)}"
out_df = df[COLUMNS].copy()

format_spec = {
    "version": "formats-v1",
    "A": "tgt_a_code: code exactly as in the dataset",
    "B": "tgt_b_canonical: docstrings/comments dropped, layout by ast.unparse",
    "C": "tgt_c_tree: full compact AST S-expression; decode with text_to_source (see notebook Cell 3)",
    "D": "tgt_d_stmt_tree: statement tree, expressions as backtick text leaves; same decoder",
    "docstring_rule": "dropped in B/C/D except refactor rows whose instruction asks to add docstrings",
    "import_rule": ("unused imports removed from B/C/D" if PRUNE_UNUSED_IMPORTS else "imports unchanged in all formats"),
    "token_budget": {"target": MAX_TARGET_TOKENS, "input": MAX_INPUT_TOKENS, "tokenizer": TOKENIZER_NAME},
    "python_version": sys.version.split()[0],
    "split": "by unit_id hash, 80/10/10",
}
table = pa.Table.from_pandas(out_df, preserve_index=False)
meta = dict(table.schema.metadata or {})
meta[b"format_spec"] = json.dumps(format_spec).encode("utf-8")
table = table.replace_schema_metadata(meta)
pq.write_table(table, OUT_FILE, compression="zstd")
print(f"Wrote {OUT_FILE}  ({OUT_FILE.stat().st_size/1e6:.1f} MB, {len(out_df)} rows, {len(COLUMNS)} columns)")

# ---- verification: read it back and re-check the contents ----
back = pd.read_parquet(OUT_FILE)
assert list(back.columns) == COLUMNS, "columns changed on the round trip"
assert len(back) == len(out_df), "row count changed"
assert back["id"].is_unique, "duplicate ids"
print("Read back: rows and columns match.")

sample = back[back.ok_c & back.ok_d & back.ok_b].sample(min(400, int(back.ok_c.sum())), random_state=0)
bad = 0
for _, r in sample.iterrows():
    try:
        b = r["tgt_b_canonical"]
        same_c = norm_dump(text_to_source(r["tgt_c_tree"])) == norm_dump(b)
        same_d = norm_dump(text_to_source(r["tgt_d_stmt_tree"])) == norm_dump(b)
        compiles = compile(b, "<b>", "exec") is not None
        bad += not (same_c and same_d and compiles)
    except Exception:
        bad += 1
print(f"Re-decoded {len(sample)} random rows from the saved file: {bad} problems (should be 0)")
assert bad == 0
if back.has_golden_io.any():
    g = back[back.has_golden_io].iloc[0]
    parsed = json.loads(g["golden_io_json"])
    print(f"golden_io_json parses back to a list of {len(parsed)} cases; golden_func = {g['golden_func']}")
print("Saved file metadata keys:", [k.decode() for k in pq.read_schema(OUT_FILE).metadata if k.startswith(b"format")])

In [ ]:
# CELL 8 — AUDIT (read this before downloading). It reports; it does not decide anything.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
d = pd.read_parquet(OUT_FILE)

print("=" * 90)
print(f"ROWS: {len(d)}   units: {d.unit_id.nunique()}   (units <= 60 lines: set by the dataset generator)")
print("=" * 90)

print("\n1) COVERAGE: what kinds of code are in the table")
print(pd.crosstab(d["unit_type"], d["task"], margins=True).to_string())
print("\n   by domain:")
print(d["domain"].value_counts().to_string())
print(f"\n   lines per unit: median {d.n_lines.median():.0f}, p90 {d.n_lines.quantile(.9):.0f}, max {d.n_lines.max()}")
print(f"   classes: {(d.unit_type=='class').sum()} rows; methods per class unit: "
      f"median {d[d.unit_type=='class'].n_methods.median():.0f}, max {d[d.unit_type=='class'].n_methods.max()}")

tags = ["abstract_base_class", "protocol", "SOLID", "dependency_injection", "repository_pattern",
        "factory_pattern", "observer_pattern", "strategy_pattern", "builder_pattern", "singleton_pattern",
        "adapter_pattern", "service_layer", "middleware", "rest_api", "routing", "endpoint", "orm",
        "pydantic", "dataclasses", "generics", "async_await", "retry", "caching", "error_handling"]
tag_counts = {t: int(d["concepts"].apply(lambda cs: t in list(cs)).sum()) for t in tags}
print("\n   design / API / DB concept tags (rows):")
print("   " + ", ".join(f"{t}={n}" for t, n in tag_counts.items() if n))
print("   NOTE: these tags come from name-matching heuristics in the dataset generator, so treat them as approximate.")

print("\n2) ROUND-TRIP CHECKS (format text -> Python -> same AST as the original)")
for k in "bcd":
    print(f"   {k.upper()}: {int(d['ok_'+k].sum())}/{len(d)}")
fails = d[~(d.ok_b & d.ok_c & d.ok_d)]
print(f"   rows with any failure: {len(fails)}" + (f"  e.g. {fails.format_errors.head(3).tolist()}" if len(fails) else ""))

print("\n3) TOKENS (CodeT5 tokenizer) and the 512-token output budget")
rows_t = []
for k, name in zip("abcd", ["A original", "B canonical", "C full tree", "D stmt tree"]):
    s = d[f"tok_{k}"].dropna().astype(float)
    rows_t.append({"format": name, "mean": s.mean(), "median": s.median(), "p90": s.quantile(.9), "max": s.max(),
                   "fits_512_%": d[f"fits512_{k}"].mean() * 100})
print(pd.DataFrame(rows_t).round(1).to_string(index=False))
print(f"   encoder input fits 512: {d.fits512_input.mean()*100:.1f}% of rows")
print(f"   rows where A, B, C and D ALL fit in 512: {int((d.fits512_a & d.fits512_b & d.fits512_c & d.fits512_d).sum())}")

print("\n   % of targets that fit in 512, by unit type:")
print((d.groupby("unit_type")[["fits512_a", "fits512_b", "fits512_c", "fits512_d"]].mean() * 100).round(1)
      .join(d.groupby("unit_type").size().rename("rows")).to_string())
print("\n   by task:")
print((d.groupby("task")[["fits512_a", "fits512_b", "fits512_c", "fits512_d"]].mean() * 100).round(1).to_string())
print("\n   by domain:")
print((d.groupby("domain")[["fits512_a", "fits512_b", "fits512_c", "fits512_d"]].mean() * 100).round(1)
      .join(d.groupby("domain").size().rename("rows")).to_string())

print("\n4) EVALUATION SUPPORT")
print(f"   rows with golden_io: {int(d.has_golden_io.sum())} ({d.has_golden_io.mean()*100:.1f}%) "
      f"-- only fully-typed top-level plain functions; classes and async code have none")
print(pd.crosstab(d["split"], d["has_golden_io"], rownames=["split"], colnames=["has_golden_io"]).to_string())

print("\n5) SAMPLES (the same unit in all four formats)")
def show(row, limit=650):
    print(f"\n--- {row['unit_type']} | {row['task']} | {row['domain']} | id={row['id']} ---")
    for k, name in zip("abcd", ["A original", "B canonical", "C full tree", "D stmt tree"]):
        txt = row[{"a": "tgt_a_code", "b": "tgt_b_canonical", "c": "tgt_c_tree", "d": "tgt_d_stmt_tree"}[k]]
        tl = row[f"tok_{k}"]
        print(f"[{name}  {tl} tokens]")
        print((txt[:limit] + (" ...[cut for display]" if txt and len(txt) > limit else "")) if txt else "(none)")
good = d[d.ok_b & d.ok_c & d.ok_d]
for utype in ("class", "function"):
    sub = good[good.unit_type == utype].sort_values("tok_c")
    if len(sub):
        show(sub.iloc[len(sub) // 4])

print("\n6) IMPORT NOISE (each unit carries its source file's import block)")
tot_i, tot_u = int(d.n_imports.sum()), int(d.n_unused_imports.sum())
with_i = d[d.n_imports > 0]
print(f"   import statements across rows: {tot_i}; unused by the unit's own code: {tot_u} ({tot_u/max(1,tot_i)*100:.1f}%)")
print(f"   rows with imports: {len(with_i)}; of those, >=50% of imports unused: "
      f"{int((with_i.n_unused_imports / with_i.n_imports >= 0.5).sum())}; ALL imports unused: "
      f"{int((with_i.n_unused_imports == with_i.n_imports).sum())}")
print(f"   share of a target's characters that are import lines: mean {d.import_char_share.mean()*100:.1f}%, "
      f"median {d.import_char_share.median()*100:.1f}%")
print(f"   PRUNE_UNUSED_IMPORTS was {'ON: ' + str(int(d.n_imports_removed.sum())) + ' names removed from B/C/D' if d.imports_pruned.any() else 'OFF: all formats keep their original imports'}")
print("   ('unused' is approximate: a name counts as used if it appears as a name or as a word inside any string literal.)")

print("\n" + "=" * 90)
print("NOTES TO CHECK BEFORE TRAINING")
for k, name in zip("bcd", ["B", "C", "D"]):
    pct = d[f"fits512_{k}"].mean() * 100
    if pct < 70:
        print(f"  - format {name}: only {pct:.1f}% of targets fit in 512 tokens; "
              f"rows that do not fit would be cut off during training.")
print("  - Inputs are identical across formats (raw code); only the target column differs.")
print("  - All units are the generator's 5-60 line snippets; larger classes/modules are not in this table.")
print("=" * 90)

In [ ]:
# CELL 9 — DOWNLOAD
from google.colab import files
print(f"{OUT_FILE}  {OUT_FILE.stat().st_size/1e6:.1f} MB")
files.download(str(OUT_FILE))